In [117]:
import requests
from datetime import datetime
import time
import json
from pprint import pprint
from dataclasses import dataclass

UPBIT_BASE_URL = "https://api.upbit.com/v1"
GET_MARKET_ALL = "/market/all"
GET_CURRENT_PRICES = "/ticker"

class UpbitAPIService:
    def __init__(self):
        self.base_url = UPBIT_BASE_URL
        self.headers = {"accept": "application/json"}

    def get_market_all(self) -> list[str]:
        try: 
            url = f"{self.base_url}/market/all"
            response = requests.get(url, headers=self.headers).json()
            all_pickers = [picker['market'] for picker in response]
            return all_pickers
        except Exception as e:
            print(f"Error fetching market data: {e}")
            return None

    def get_current_prices_api(self, markets: list[str]) -> dict:
        try:
            url = f"{self.base_url}/ticker"
            response: list[dict] = requests.get(url, headers=self.headers, params={"markets": ",".join(markets)}).json()
            response_dict = {picker['market']: float(picker['opening_price']) for picker in response}
            return response_dict
        except Exception as e:
            print(f"Error fetching current prices: {e}")
            return None

@dataclass
class Holding:
    market: str
    quantity: float
    current_price: float = 0.0

    def calculate_total_price(self) -> float:
        return self.quantity * self.current_price
    
    def set_current_price(self, price: float):
        self.current_price = price

@dataclass
class Portfolio:
    holdings: list[Holding]

    def calculate_total_price(self):
        return sum([holding.calculate_total_price() for holding in self.holdings])
            
    def analyze_portfolio(self):

        info = []
        for holding in self.holdings:
            info.append({
                "market": holding.market,
                "quantity": holding.quantity,
                "current_price": holding.current_price,
                "total_price": holding.calculate_total_price(),
                "portion": holding.calculate_total_price() / self.calculate_total_price() * 100,
            })

        return {
            "info": info,
            "total_portfolio_value": self.calculate_total_price()
        }

In [118]:
upbit_service = UpbitAPIService()

price_map = upbit_service.get_current_prices_api(["KRW-BTC", "KRW-ETH", "KRW-XRP", "KRW-DOGE", "KRW-ADA", "KRW-SOL"])
holdings = [
    Holding(market="KRW-BTC", quantity=0.001, current_price=price_map["KRW-BTC"]),
    Holding(market="KRW-ETH", quantity=0.01, current_price=price_map["KRW-ETH"]),
    Holding(market="KRW-XRP", quantity=100, current_price=price_map["KRW-XRP"]),
    Holding(market="KRW-DOGE", quantity=1000, current_price=price_map["KRW-DOGE"]),
    Holding(market="KRW-ADA", quantity=1000, current_price=price_map["KRW-ADA"]),
    Holding(market="KRW-SOL", quantity=1, current_price=price_map["KRW-SOL"])
]
my_portfolio = Portfolio(holdings=holdings)
analysis = my_portfolio.analyze_portfolio()
# 표 헤더
print("=" * 80)
print(f"{'마켓':<15} {'수량':<15} {'현재가':<15} {'총가치':<15} {'비중':<10}")
print("=" * 80)

# 데이터 출력
for holding in analysis['info']:
    print(f"{holding['market']:<15} "
          f"{holding['quantity']:>14.8f}개 "
          f"{holding['current_price']:>13,.0f}원 "
          f"{holding['total_price']:>13,.0f}원 "
          f"{holding['portion']:>8.0f}%")

print("=" * 80)
print(f"총 포트폴리오 가치: {analysis['total_portfolio_value']:.0f}원")

마켓              수량              현재가             총가치             비중        
KRW-BTC             0.00100000개   158,129,000원       158,129원        7%
KRW-ETH             0.01000000개     5,817,000원        58,170원        2%
KRW-XRP           100.00000000개         4,016원       401,600원       17%
KRW-DOGE         1000.00000000개           335원       335,000원       14%
KRW-ADA          1000.00000000개         1,142원     1,142,000원       48%
KRW-SOL             1.00000000개       295,600원       295,600원       12%
총 포트폴리오 가치: 2390499원
